<style>
    .jp-RenderedHTMLCommon {
        font-size: 12px !important;
    }
    .jp-RenderedHTMLCommon h1 {
        font-size: 14px !important;
    }
    .jp-RenderedHTMLCommon h2 {
        font-size: 13px !important;
    }
    .jp-RenderedHTMLCommon h3 {
        font-size: 12px !important;
    }
    .jp-RenderedHTMLCommon p,
    .jp-RenderedHTMLCommon li {
        font-size: 11px !important;
        line-height: 1.4 !important;
    }
    .jp-RenderedHTMLCommon code {
        font-size: 11px !important;
    }
</style>

# ============================================================
# CELL 0: IMPORTS - Bringing in the tools we need
# ============================================================

## `import tensorflow as tf`
- `import` = tells Python to load a library/package into memory so we can use it
- `tensorflow` = Google's deep learning library (makes neural networks easy to build)
- `as tf` = gives it a nickname 'tf' so we write `tf.something()` instead of `tensorflow.something()`
  - This is just a convention, you could do `import tensorflow as banana` but that's weird

## `from tensorflow.keras.preprocessing.image import ImageDataGenerator`
- `from ... import ...` = imports only ONE specific thing from a library (not the whole library)
- `keras` = high-level API inside TensorFlow (makes building neural networks easier, like LEGO blocks)
- `preprocessing.image` = tools for preparing images before feeding to the model
- `ImageDataGenerator` = a class that:
  1. Loads images from folders automatically
  2. Applies data augmentation (random flips, zooms, shears) to create variety
  3. Splits images into batches (groups) for efficient training
  4. Automatically labels images based on folder names (e.g., 'cats' folder = label 0)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ============================================================
# CELL 1: CHECK TENSORFLOW VERSION
# ============================================================

## `tf.__version__`
- `__version__` = a special built-in attribute (double underscore = "dunder") that stores version info
- Returns a string like '2.21.0' (major.minor.patch)
- Useful to check: (a) TF is installed correctly, (b) which version you have
- Different versions have different APIs, so knowing your version helps debug issues

In [ ]:
tf.__version__

# ============================================================
# CELL 2: DATA PREPROCESSING
# Before feeding images to a neural network, we must:
#   1. Load images from disk
#   2. Resize them all to the same size (64x64)
#   3. Normalize pixel values (0-255 -> 0-1)
#   4. Apply data augmentation to training set (prevents overfitting)
# ============================================================

---

## `TRAIN_DIR = 'C:/Users/SCS/Desktop/...'
- This is just a string variable storing the folder path
- Windows uses backslashes `\` but Python strings can use forward slashes `/` too
- We store it in a variable so if the path changes, we only update it in ONE place

---

## `ImageDataGenerator(rescale=1./255, shear_range=0.2, ...)`
- Creates a "generator" object that will load and transform images on-the-fly
- **`rescale=1./255`** = Normalization:
  - Each pixel has a value from 0 to 255 (8-bit image)
  - Dividing by 255 converts it to 0.0 to 1.0 range
  - WHY? Neural networks train faster and more accurately with small numbers (0-1)
  - Example: pixel value 200 becomes 200/255 = 0.784
- **`shear_range=0.2`** = Shear transformation:
  - Shear = tilting/slanting the image (like pushing the top while holding the bottom)
  - 0.2 = up to 20% slant
  - WHY? Makes the model robust to slightly tilted images
  - Example: A slightly tilted cat photo should still be recognized as a cat
- **`zoom_range=0.2`** = Random zoom:
  - Randomly zooms in or out by up to 20%
  - WHY? Teaches the model to recognize objects at different sizes/distances
- **`horizontal_flip=True`** = Random horizontal flip:
  - Mirrors the image left-to-right randomly
  - WHY? A cat facing left is still a cat facing right
  - Note: We don't flip VERTICALLY because upside-down cats/dogs don't make sense

---

## `train_datagen.flow_from_directory(TRAIN_DIR, target_size=(64,64), ...)`
- **`flow_from_directory()`** = the magic method that does everything:
  1. Scans the TRAIN_DIR folder
  2. Finds subfolders (e.g., 'cats/', 'dogs/') - each subfolder = one class
  3. Reads ALL images from each subfolder
  4. Resizes each image to `target_size=(64,64)` - width=64px, height=64px
  5. Groups them into batches of `batch_size=32` (32 images per group)
  6. Assigns labels: 'cats' folder -> label 0, 'dogs' folder -> label 1
  7. Returns an iterator that yields (batch_of_images, batch_of_labels) one at a time
- **`target_size=(64,64)`** = all images must be same size for the neural network
  - Neural networks require fixed input dimensions
  - Even if original image is 1024x768, it gets resized to 64x64
- **`batch_size=32`** = process 32 images at once (not 1 by 1, not all 8000 at once)
  - Too small = slow training
  - Too large = runs out of memory (RAM/GPU)
  - 32 is a good default balance
- **`class_mode='binary'`** = tells Keras this is a 2-class problem (cat vs dog)
  - 'binary' = 2 classes -> output layer has 1 neuron + sigmoid
  - 'categorical' = 3+ classes -> output layer has N neurons + softmax

---

## `test_datagen = ImageDataGenerator(rescale=1./255)`
- For test data, we ONLY rescale (no augmentation!)
- WHY? We want to test the model on REAL images, not augmented/fake ones
- If we augment test data, we're not measuring how well the model performs in real life

In [ ]:
# ============================================================
# DATA PREPROCESSING
# ============================================================

TRAIN_DIR = 'C:/Users/SCS/Desktop/MetPi_AI/Copy of dataset/dataset/training_set'
TEST_DIR = 'C:/Users/SCS/Desktop/MetPi_AI/Copy of dataset/dataset/test_set'

# Training data generator WITH augmentation (to prevent overfitting)
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalize: divide all pixel values by 255 (0-255 -> 0.0-1.0)
    shear_range=0.2,         # Randomly tilt images up to 20%
    zoom_range=0.2,          # Randomly zoom in/out by up to 20%
    horizontal_flip=True     # Randomly flip images left-right
)

# Load training images from folder structure:
# training_set/
#   cats/   -> label 0
#   dogs/   -> label 1
training_set = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(64, 64),    # Resize every image to 64x64 pixels
    batch_size=32,           # Yield 32 images at a time
    class_mode='binary'      # 2 classes: cats=0, dogs=1
)

# Test data generator - NO augmentation (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load test images (same folder structure, no augmentation)
test_set = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

# ============================================================
# CELL 3: BUILD THE CNN (Convolutional Neural Network)
# A CNN works like a series of filters that extract features:
#   - Early layers detect SIMPLE features (edges, colors, textures)
#   - Deeper layers detect COMPLEX features (eyes, ears, fur patterns)
#   - Final layers recognize WHOLE OBJECTS (cat face, dog face)
# ============================================================

---

## `cnn = tf.keras.models.Sequential()`
- `tf.keras.models` = Keras module for creating neural network models
- `Sequential()` = creates an empty model where you add layers ONE BY ONE (in order)
- Think of it like an empty sandwich: you add bread, then lettuce, then meat, etc.
- Alternative: `Functional API` = more complex models with branches/connections

---

## `cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64,64,3]))`
- **`cnn.add()`** = adds a new layer to the model (like stacking LEGO blocks)
- **`Conv2D`** = Convolutional layer for 2D images:
  - **`filters=32`** = creates 32 different feature detectors (also called kernels)
    - Each filter learns to detect ONE specific pattern (e.g., vertical edge, horizontal edge, corner, color blob)
    - 32 filters = model can detect 32 different patterns simultaneously
  - **`kernel_size=3`** = each filter is a 3x3 pixel matrix
    - The filter slides over the image, comparing the 3x3 patch at each position
    - Output: a "feature map" showing where that pattern was found
  - **`activation='relu'`** = Rectified Linear Unit:
    - Formula: `relu(x) = max(0, x)`
    - If a value is positive, keep it. If negative, make it 0.
    - WHY? Adds non-linearity (so the model can learn complex patterns)
    - Without activation, the whole network is just a linear equation (useless)
  - **`input_shape=[64, 64, 3]`** = tells the model what to expect:
    - 64 = height in pixels
    - 64 = width in pixels
    - 3 = color channels (Red, Green, Blue = RGB)
    - Only needed for the FIRST layer (Keras figures out the rest automatically)

---

## `cnn.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2))`
- **`MaxPooling2D`** = reduces the size of feature maps:
  - **`pool_size=2`** = look at 2x2 blocks, take the MAXIMUM value from each block
    - Example: [[1,3],[2,4]] -> 4 (the largest number wins)
  - **`strides=2`** = move the 2x2 window 2 pixels at a time (no overlap)
  - RESULT: feature map size is cut in HALF (64x64 -> 32x32)
  - WHY?
    1. Reduces computation (smaller data = faster training)
    2. Keeps the MOST IMPORTANT features (the maximum values)
    3. Makes the model robust to small shifts in position

---

## Second Conv + Pool (Layer 2)
- Same as Layer 1 but NO `input_shape` needed (Keras already knows)
- Deeper layers learn MORE COMPLEX features
  - Layer 1: edges, colors, simple textures
  - Layer 2: combinations of those (eyes, ears, noses, fur patterns)

---

## `cnn.add(tf.keras.layers.Flatten())`
- **`Flatten()`** = converts 2D feature maps into a 1D long vector
- After pooling, data shape is (14, 14, 32) = a 3D volume
- Flatten makes it (6272,) = one long line of numbers
- WHY? Dense (fully connected) layers need 1D input, not 2D
- Think of it like unfolding a matrix into a single row

---

## `cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))`
- **`Dense`** = fully connected layer (every neuron connects to ALL neurons in previous layer)
- **`units=128`** = 128 neurons in this layer
  - More neurons = more learning capacity (but slower, risk of overfitting)
  - 128 is a common choice for medium-sized problems
- **`activation='relu'`** = same ReLU activation (adds non-linearity)
- Each neuron has weights (numbers) that get adjusted during training
- The network learns the BEST weights to classify cats vs dogs

---

## `cnn.add(tf.keras.layers.Dropout(0.2))`
- **`Dropout`** = randomly turns OFF 20% of neurons during each training step
- **`0.2`** = 20% of neurons are randomly set to zero
- WHY?
  - Prevents overfitting (the model memorizing training data instead of learning patterns)
  - Forces the network to not rely on any single neuron
  - Like studying for an exam: if you rely on ONE note, you fail if you lose it
  - Dropout makes the model more robust by forcing it to learn redundant representations
- NOTE: Dropout is ONLY active during training, NOT during prediction

---

## `cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))`
- **OUTPUT LAYER** - the final decision maker
- **`units=1`** = ONE neuron (because we have 2 classes: cat or dog)
  - 1 neuron with sigmoid = binary classification
  - 2+ neurons with softmax = multi-class classification
- **`activation='sigmoid'`** = squashes output to a value between 0 and 1:
  - Formula: `sigmoid(x) = 1 / (1 + e^(-x))`
  - Output close to 0 = likely CAT
  - Output close to 1 = likely DOG
  - Output around 0.5 = uncertain

---

## `cnn.summary()`
- Prints a nice table showing:
  - Each layer name and type
  - Output shape after each layer
  - Number of trainable parameters (weights) in each layer
  - Total parameters at the bottom

In [ ]:
# ============================================================
# BUILDING THE CNN
# ============================================================

# Sequential() = empty model, we add layers one by one
cnn = tf.keras.models.Sequential()

# --- LAYER 1: First Convolution + Pooling ---
# Conv2D = slides 32 filters (3x3 each) over the image to detect features like edges/colors
# relu = max(0, x) -> keeps positive values, sets negatives to 0 (adds non-linearity)
# input_shape=[64,64,3] = 64x64 RGB image (3 color channels)
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

# MaxPool2D = takes 2x2 blocks, keeps the MAX value -> reduces size by half (64->32)
# Keeps most important features, reduces computation
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2))

# --- LAYER 2: Second Convolution + Pooling ---
# Deeper layers learn more complex features (eyes, ears, fur patterns)
# No input_shape needed here, Keras already knows from Layer 1
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2))

# --- FLATTENING ---
# Converts 2D feature maps (14x14x32) into 1D vector (6272 numbers)
# Required because Dense layers need flat input
cnn.add(tf.keras.layers.Flatten())

# --- FULL CONNECTION ---
# Dense(128) = 128 neurons, each connected to ALL 6272 inputs from flatten layer
# Each neuron learns to combine features to make decisions
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

# Dropout(0.2) = randomly turns off 20% of neurons during training
# Prevents overfitting (memorizing training data instead of learning patterns)
cnn.add(tf.keras.layers.Dropout(0.2))

# --- OUTPUT LAYER ---
# Dense(1) = single neuron output
# sigmoid = squashes output to 0.0 - 1.0 (probability)
# < 0.5 = CAT, >= 0.5 = DOG
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# summary() prints model architecture: layers, output shapes, number of parameters
cnn.summary()

# ============================================================
# CELL 4: COMPILE THE MODEL
# Before training, we must tell the model HOW to learn:
#   - Which optimizer to use (how to adjust weights)
#   - Which loss function to use (how to measure errors)
#   - Which metrics to track (how to measure success)
# ============================================================

---

## `cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])`

### `optimizer='adam'`
- An optimizer controls HOW the model updates its weights after each batch
- **Adam** = Adaptive Moment Estimation:
  - The most popular optimizer (works well in most cases)
  - Automatically adjusts learning rate for each weight
  - Combines benefits of two other optimizers: RMSprop + Momentum
  - Think of it like: the model learns FAST when it's wrong, SLOWLY when it's close
- Alternatives: 'sgd' (slower), 'rmsprop', 'adagrad', etc.

### `loss='binary_crossentropy'`
- Loss function = measures HOW WRONG the model's prediction is
- **Binary Cross-Entropy** = standard loss for 2-class problems:
  - Formula: `loss = -[y * log(y_pred) + (1-y) * log(1-y_pred)]`
  - When model predicts correctly -> loss is close to 0
  - When model predicts wrong -> loss is HIGH
  - The model's goal: MINIMIZE this loss
- Alternatives:
  - 'categorical_crossentropy' = for 3+ classes
  - 'mse' (mean squared error) = for regression (predicting numbers)

### `metrics=['accuracy']`
- Metrics = what we MONITOR during training (doesn't affect learning, just for display)
- **accuracy** = percentage of correct predictions
  - accuracy = correct_predictions / total_predictions
  - Example: 80 out of 100 correct = 80% accuracy
- We watch accuracy to see if the model is improving

In [ ]:
# ============================================================
# COMPILE THE MODEL
# ============================================================

cnn.compile(
    optimizer='adam',                # Adam optimizer: adjusts weights intelligently, fast convergence
    loss='binary_crossentropy',      # Loss for 2 classes: penalizes wrong predictions heavily
    metrics=['accuracy']             # Track: what % of predictions are correct
)

# ============================================================
# CELL 5: TRAIN THE MODEL
# This is where the actual learning happens!
# The model looks at images, makes guesses, checks how wrong it is,
# and adjusts its weights to get better next time.
# ============================================================

---

## `history = cnn.fit(x=training_set, validation_data=test_set, epochs=25)`

### `cnn.fit()`
- The main training function
- Loops through the training data multiple times, adjusting weights each time

### `x=training_set`
- `x` = the input data (training images + labels)
- `training_set` = the ImageDataGenerator iterator we created earlier
- It will automatically pull batches of 32 images, feed them to the model, get gradients, update weights

### `validation_data=test_set`
- `validation_data` = data used to CHECK performance after each epoch
- The model NEVER trains on this data - it only evaluates
- If training accuracy is high but validation accuracy is low -> overfitting
- If both are low -> underfitting (need more epochs or a better model)

### `epochs=25`
- **`epoch`** = one complete pass through the ENTIRE training dataset
- 25 epochs = the model will see every training image 25 times
- Each epoch, the model gets better (loss decreases, accuracy increases)
- Too few epochs = model doesn't learn enough (underfitting)
- Too many epochs = model memorizes training data (overfitting)

### `history = ...`
- `history` object stores the training metrics (loss, accuracy) for each epoch
- You can later plot these to see training progress
- Example: `history.history['accuracy']` = list of accuracy values per epoch

### What happens during training:
1. Model gets a batch of 32 images
2. Makes predictions (forward pass)
3. Calculates loss (how wrong it was)
4. Adjusts weights to reduce loss (backward pass / backpropagation)
5. Repeats for ALL batches = 1 epoch
6. After each epoch, evaluates on test_set
7. Repeats for 25 epochs

In [ ]:
# ============================================================
# TRAIN THE MODEL
# ============================================================

history = cnn.fit(
    x=training_set,          # Training data: 8000 images of cats and dogs
    validation_data=test_set, # Test data: 2000 images to check performance after each epoch
    epochs=25                 # Number of times the model sees the entire training set
)

# ============================================================
# CELL 6: MAKE A SINGLE PREDICTION
# Now that the model is trained, let's use it!
# Load ONE image and ask: "Is this a cat or a dog?"
# ============================================================

---

## `import numpy as np`
- `numpy` = Python's most important library for math/arrays
- `as np` = nickname so we write `np.something()`
- Neural networks work with NUMPY ARRAYS, not regular Python lists
- We need numpy to:
  - Convert images to arrays
  - Add extra dimensions (batch dimension)
  - Do mathematical operations

---

## `from tensorflow.keras.preprocessing import image`
- Imports the `image` module from Keras preprocessing
- This module has helper functions for loading and manipulating images
- We use it to load an image file and convert it to a numpy array

---

## `img = image.load_img('path/to/image.jpg', target_size=(64, 64))`
- **`image.load_img()`** = loads an image from disk and resizes it
- Returns a **PIL Image object** (not a numpy array yet!)
- `target_size=(64, 64)` = resizes to 64x64 pixels (same as training)
- The path must point to an actual image file (.jpg, .png, etc.)
- If file doesn't exist -> FileNotFoundError

---

## `img_array = image.img_to_array(img)`
- **`image.img_to_array()`** = converts PIL Image to a numpy array
- Before: PIL Image object (can't do math with it)
- After: numpy array of shape (64, 64, 3) = 64 rows, 64 columns, 3 color channels
- Each pixel is now a number from 0.0 to 255.0

---

## `img_array = np.expand_dims(img_array, axis=0)`
- **`np.expand_dims()`** = adds an EXTRA dimension to the array
- Before shape: (64, 64, 3) = one image
- After shape: (1, 64, 64, 3) = a batch of one image
- **WHY?** The model was trained on BATCHES of images, not single images
  - Training: model receives (32, 64, 64, 3) = 32 images at once
  - Prediction: model expects (N, 64, 64, 3) where N = number of images
  - For 1 image: N = 1, so we add a dimension at axis=0
- **`axis=0`** = add dimension at position 0 (the batch dimension)
  - axis=0: (64,64,3) -> (1,64,64,3) 
  - axis=1: (64,64,3) -> (64,1,64,3) - WRONG!
- Without this, you'd get an error: "Input 0 of layer expects 4 dimensions, but got 3"

---

## `prediction = cnn.predict(img_array)`
- **`cnn.predict()`** = feeds the image through the trained model
- Returns a numpy array with the prediction
- Output shape: (1, 1) = batch of 1 image, 1 output neuron
- `prediction[0][0]` = the actual probability value (a float between 0 and 1)

---

## `if prediction[0][0] > 0.5: print('DOG') else: print('CAT')`
- The sigmoid output is a probability:
  - `> 0.5` = model thinks it's a DOG (more than 50% confident)
  - `<= 0.5` = model thinks it's a CAT
- `prediction[0][0]` indexing:
  - `prediction` = array of shape (1, 1)
  - `prediction[0]` = first (and only) batch = array of shape (1,)
  - `prediction[0][0]` = the actual float value (e.g., 0.87)

In [ ]:
# ============================================================
# SINGLE PREDICTION
# ============================================================

import numpy as np                                    # numpy = math/arrays library
from tensorflow.keras.preprocessing import image      # image = helper to load/manipulate images

# Load an image from disk and resize to 64x64 (same as training)
# Returns a PIL Image object
img = image.load_img(
    'C:/Users/SCS/Desktop/MetPi_AI/Copy of dataset/dataset/single_prediction/cat_or_dog_1.jpg',
    target_size=(64, 64)
)

# Convert PIL Image -> numpy array (64, 64, 3)
img_array = image.img_to_array(img)

# Add batch dimension: (64, 64, 3) -> (1, 64, 64, 3)
# Model expects batches, even for a single image
img_array = np.expand_dims(img_array, axis=0)

# Predict: model outputs a probability (0.0 to 1.0)
# < 0.5 = CAT, >= 0.5 = DOG
prediction = cnn.predict(img_array)

# Print result
if prediction[0][0] > 0.5:
    print('Prediction: DOG')
else:
    print('Prediction: CAT')